# 02 — YOLOv8-nano Logo Detector Training
## Brand Vision — Local RTX 4050 or Kaggle P100

**Prerequisite:** Run `01_data_preparation.ipynb` first.

- **Local (RTX 4050):** `batch=16`, ~45–90 min for 50 epochs
- **Kaggle P100:** `batch=32`, ~25–40 min for 60 epochs

Update `DATA_YAML_PATH` below before running.

In [5]:
# Cell 1 — Install (run on Kaggle; skip locally — already in venv)
# !pip install ultralytics -q

In [6]:
# Cell 2 — Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.4 GB


In [7]:
# Cell 3 — Config
from pathlib import Path

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
# Local:  absolute path to your logo_data.yaml
# Kaggle: "/kaggle/working/yolo_logo_detection/logo_data.yaml"
DATA_YAML_PATH = str(
    Path("datasets/yolo_logo_detection/logo_data.yaml").resolve()
)
# ─────────────────────────────────────────────────────────────────────────────

# Detect platform: Kaggle has 16GB P100, local has 6GB RTX 4050
import os
ON_KAGGLE = os.path.exists("/kaggle")
BATCH     = 32 if ON_KAGGLE else 16   # max 16 on local 6GB VRAM
EPOCHS    = 60 if ON_KAGGLE else 50
CACHE     = "ram" if ON_KAGGLE else True

print(f"Running on: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"data yaml:  {DATA_YAML_PATH}")
print(f"batch={BATCH}, epochs={EPOCHS}")

Running on: local
data yaml:  C:\Projects\INT428 DL CA01\brand-vision\training\datasets\yolo_logo_detection\logo_data.yaml
batch=16, epochs=50


In [8]:
# Cell 4 — Train YOLOv8-nano
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Downloads COCO pretrained weights on first run

results = model.train(
    data      = DATA_YAML_PATH,
    epochs    = EPOCHS,
    imgsz     = 640,
    batch     = BATCH,
    device    = 0,             # GPU 0
    project   = "runs/logo_detector",
    name      = "v1",
    pretrained= True,
    patience  = 10,            # Early stopping
    lr0       = 0.01,
    lrf       = 0.001,
    optimizer = "AdamW",
    amp       = True,          # AMP — saves VRAM
    cache     = CACHE,
    workers   = 4,
    save      = True,
    save_period = 10,
    mosaic    = 1.0,
    fliplr    = 0.5,
    degrees   = 5.0,
    warmup_epochs = 3,
    verbose   = True,
)

print(f"\nTraining complete!")
print(f"Best weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.26  Python-3.14.3 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Projects\INT428 DL CA01\brand-vision\training\datasets\yolo_logo_detection\logo_data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v12, nbs=64, nms=False, opset=N

In [9]:
# Cell 5 — Validate
best_pt = f"{results.save_dir}/weights/best.pt"
model_best = YOLO(best_pt)
metrics = model_best.val(data=DATA_YAML_PATH)

map50 = metrics.box.map50
print(f"mAP50:     {map50:.4f}  (target: >0.60, ideal: >0.75)")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

if map50 >= 0.60:
    print("\n✅ Phase 2 acceptance criteria met! Proceed to ONNX export.")
else:
    print("\n⚠ mAP50 below 0.60 — consider training more epochs or checking dataset.")

Ultralytics 8.4.26  Python-3.14.3 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 76.336.1 MB/s, size: 15.3 KB)
val: Scanning C:\Projects\INT428 DL CA01\brand-vision\training\datasets\yolo_logo_detection\labels\val.cache... 258 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 258/258 72.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 2.2it/s 7.9s0.2s
                   all        258        311       0.83      0.814      0.873      0.583
Speed: 1.8ms preprocess, 4.9ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\Projects\INT428 DL CA01\brand-vision\runs\detect\val
mAP50:     0.8728  (target: >0.60, ideal: >0.75)
mAP50-95:  0.5825
Precision: 0.8297
Recall:    0.8135

✅ Phase 2 acceptance criteria met! Proceed to ONNX export.


In [10]:
# Cell 6 — Export to ONNX
model_best.export(format="onnx", imgsz=640, dynamic=True, simplify=True)
print(f"ONNX saved to: {results.save_dir}/weights/best.onnx")
print("Next: copy best.onnx to backend/weights/logo_detector.onnx")

Ultralytics 8.4.26  Python-3.14.3 torch-2.11.0+cu126 CPU (Intel Core 5 210H)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'C:\Projects\INT428 DL CA01\brand-vision\runs\detect\runs\logo_detector\v12\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/209.6 MB ? eta -:--:--
    --------------------------------------- 2.9/209.6 MB 16.4 MB/s eta 0:00:13
   - -------------------------------------- 6.0/209.6 MB 15.3 MB/s eta 0:00:14
   - -------------------------------------- 8.9/209.6 MB 14.7 MB/s eta 0:00:14
   -- ------------------------------------- 11.3/209.6 MB 14.0 MB/s eta 0:00:15
   -- ------------------------------------- 13.6/209.6 MB 13.4 MB/s eta 0:00:15
 